# AgentCore Memory를 사용하는 LlamaIndex - 학술 연구 도우미(장기 메모리)

## 소개

이 Notebook에서는 Amazon Bedrock AgentCore Memory 기능을 LlamaIndex와 통합하여 여러 연구 session에 걸쳐 **장기 메모리**를 유지하는 학술 연구 도우미를 만드는 방법을 살펴봅니다. 이를 통해 도우미는 수주 또는 수개월간 진행되는 연구에서 지식을 축적할 수 있습니다.

## 아키텍처 개요

![LlamaIndex AgentCore 장기 메모리 아키텍처](LlamaIndex-AgentCore-LTM-Arch.png)

## 튜토리얼 세부 정보

**튜토리얼 세부 정보:**
- **튜토리얼 유형**: Session 간 장기 메모리
- **Agent 사용 사례**: 학술 연구 도우미
- **Agentic Framework**: LlamaIndex
- **LLM model**: Anthropic Claude 3.7 Sonnet
- **튜토리얼 구성 요소**: AgentCore 장기 메모리, LlamaIndex Agent, 연구 Tool
- **예제 난이도**: 고급

## 비즈니스 가치

**Enterprise Research Intelligence**: 조직의 지식을 축적하고 연구 발전 과정을 추적하며 여러 project와 기간에 걸쳐 종합적인 학술 맥락을 유지하는 지속형 AI 메모리로 연구 workflow를 혁신합니다.

**주요 전문적 이점:**
- **연구 연속성**: 연구 단계와 팀 구성원 간에 지식을 원활하게 이전
- **조직 메모리**: 중요한 연구 insight, 방법론, 결과를 영구 보존
- **Project 간 Intelligence**: 여러 연구 initiative에서 pattern과 연관성 식별
- **우수한 연구비 제안서**: 이전 연구 데이터를 활용하여 설득력 있는 연구비 신청서 작성
- **학술 협업**: 여러 해에 걸친 공동 연구 project의 상세 맥락 유지
- **출판 전략**: 전략적인 출판 계획을 위해 연구 주제와 인용 network 추적

## 장기 메모리 구성

**기술 설정**: 이 튜토리얼에서는 Semantic Strategy가 적용된 AgentCore Memory를 사용하여 데이터를 12개월간 보존합니다.
- **Memory 유형**: Insight를 자동으로 추출하는 semantic strategy
- **보존 기간**: 연구 연속성을 위한 365일 event 만료 기간
- **Session 간 구성**: 동일한 actor_id + memory_id, 연구 기간별로 서로 다른 session_id
- **검색 기능**: 전체 연구 기록을 semantic search하는 기본 제공 memory 검색 tool

## 기술 개요

**주요 장기 메모리 구성 요소:**
1. **Semantic Strategy 구성**: SemanticStrategy를 사용하여 insight를 자동 추출하고 365일간 보존
2. **Session 간 지속성**: 동일한 actor_id + memory_id와 기간별로 다른 session_id를 사용하여 지식 연속성 구현
3. **Custom Memory 검색 Tool**: AgentCore 기본 search_long_term_memories()를 LlamaIndex FunctionTool로 wrapping
4. **Semantic 처리 Pipeline**: 대화 event를 semantic memory로 변환하기 위해 90초 대기
5. **동적 Session 관리**: 유연한 session 처리를 위해 memory.context.session_id 사용

**다음 내용을 학습합니다:**

- 여러 연구 session에 걸쳐 지속되는 AgentCore Memory 생성
- 시간에 따라 연구 지식 축적
- 전체 연구 기록을 대상으로 semantic search 구현
- 연구 발전 과정과 전문성 향상 추적
- Session 간 메모리 지속성 및 검색 테스트

## 사전 요구 사항

- Python 3.10+
- 적절한 권한이 있는 AWS account
- AgentCore Memory 권한이 있는 AWS IAM role:
  - `bedrock-agentcore:CreateMemory`
  - `bedrock-agentcore:CreateEvent`
  - `bedrock-agentcore:ListEvents`
  - `bedrock-agentcore:RetrieveMemories`
- Amazon Bedrock model에 대한 액세스

## 1단계: Dependency 설치 및 설정

In [ ]:
# Semantic strategy toolkit을 포함한 필수 library 설치
%pip install llama-index-memory-bedrock-agentcore llama-index-llms-bedrock-converse boto3 bedrock-agentcore-starter-toolkit

In [ ]:
# 필요한 component import
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore_starter_toolkit.operations.memory.models.strategies.semantic import (
    SemanticStrategy,
)
from llama_index.memory.bedrock_agentcore import AgentCoreMemory, AgentCoreMemoryContext
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool
from datetime import datetime
import os

print("✅ All dependencies imported successfully!")

## 2단계: AgentCore Memory 구성

장기 연구 지식을 위한 AgentCore Memory resource를 생성하거나 가져옵니다.

In [ ]:
# 장기 지속성을 위해 Semantic Strategy가 적용된 AgentCore Memory 생성
region = os.getenv("AWS_REGION", "us-east-1")
memory_manager = MemoryManager(region_name=region)

try:
    # Insight 자동 추출용 semantic strategy로 memory 생성
    memory = memory_manager.get_or_create_memory(
        name=f"AcademicResearchSemantic_{int(datetime.now().timestamp())}",
        strategies=[SemanticStrategy(name="researchLongTermMemory")],
        event_expiry_days=365,  # 연구 record를 12개월간 보존
    )
    memory_id = memory.get("id")
    print(f"✅ Created Semantic Memory: {memory_id}")
    print(f"   Status: {memory.get('status')}")
    print(f"   Strategies: {[s.get('name') if isinstance(s, dict) else str(s) for s in memory.get('strategies', [])]}")

    # Memory가 ACTIVE 상태가 될 때까지 대기
    if memory.get("status") != "ACTIVE":
        print(f"\n⏳ Waiting for memory to become ACTIVE (currently {memory.get('status')})...")
        import time

        max_wait = 300  # 최대 5분
        waited = 0
        while waited < max_wait:
            time.sleep(10)
            waited += 10
            # 상태 확인
            current_memory = memory_manager.get_memory(memory_id)
            status = current_memory.get("status")
            print(f"   [{waited}s] Status: {status}")
            if status == "ACTIVE":
                print(f"✅ Memory is now ACTIVE! (took {waited} seconds)")
                break
        else:
            print(f"⚠️  Memory still not ACTIVE after {max_wait}s. Proceeding anyway...")

except Exception as e:
    print(f"❌ Error creating memory: {e}")
    memory_id = "your-memory-id-here"  # 기존 memory ID로 교체

## 3단계: 연구 Tool 구현

학술 연구 작업을 위한 전문 tool을 정의합니다.

In [ ]:
def save_paper_summary(title: str, authors: str, key_findings: str) -> str:
    """Save a research paper summary with title, authors, and key findings"""
    print(f"📄 Saved paper: {title} by {authors}")
    return f"Successfully saved paper summary for '{title}'"


def track_research_topic(topic: str, status: str) -> str:
    """Track research topic progress with current status"""
    print(f"🔬 Tracking research topic: {topic} (Status: {status})")
    return f"Now tracking research topic: {topic} with status {status}"


def save_research_finding(finding: str, confidence: str) -> str:
    """Save a research finding with confidence level"""
    print(f"💡 Research finding saved with {confidence} confidence")
    return f"Saved research finding with {confidence} confidence level"


def update_research_status(topic: str, new_status: str, notes: str) -> str:
    """Update research topic status with notes"""
    print(f"📊 Updated {topic} status to: {new_status}")
    return f"Updated research status for {topic}"


def log_research_milestone(period: str, milestone: str, details: str) -> str:
    """Log a research milestone with period and detailed progress"""
    print(f"🎯 {period} milestone: {milestone}")
    return f"Logged milestone for {period}: {milestone} - {details}"


def track_research_metrics(metric_type: str, value: str, source: str, period: str) -> str:
    """Track specific research metrics with source and timeline"""
    print(f"📊 {period}: {metric_type} = {value} (from {source})")
    return f"Tracked {metric_type}: {value} from {source} in {period}"


def save_research_insight(insight: str, period: str, connections: str) -> str:
    """Save research insights with connections to previous work"""
    print(f"💡 {period} insight: {insight[:50]}...")
    return f"Saved {period} insight with connections: {connections}"


# Agent용 tool object 생성
research_tools = [
    FunctionTool.from_defaults(fn=save_paper_summary),
    FunctionTool.from_defaults(fn=track_research_topic),
    FunctionTool.from_defaults(fn=save_research_finding),
    FunctionTool.from_defaults(fn=update_research_status),
    FunctionTool.from_defaults(fn=log_research_milestone),
    FunctionTool.from_defaults(fn=track_research_metrics),
    FunctionTool.from_defaults(fn=save_research_insight),
]

print("✅ Research tools created!")

## 3b단계: Memory 검색 Tool 추가

Agent가 장기 메모리를 검색할 수 있는 tool을 생성합니다.

In [ ]:
def create_memory_retrieval_tool(memory_id: str, actor_id: str, region: str):
    """에이전트가 자체 장기 메모리를 검색하는 도구를 생성합니다."""

    def search_long_term_memory(query: str) -> str:
        """Search long-term memory for relevant research information.

        Use this tool when you need to recall:
        - Previous research papers and findings
        - Research topics and their status
        - Metrics and insights from past work
        - Research milestones and progress

        Args:
            query: Search query describing what information you need

        Returns:
            Relevant information from long-term memory
        """
        try:
            from bedrock_agentcore.memory.session import MemorySessionManager

            # Session manager 생성
            session_manager = MemorySessionManager(memory_id=memory_id, region_name=region)

            # Semantic strategy namespace에서 장기 메모리 검색
            results = session_manager.search_long_term_memories(
                query=query,
                namespace_prefix="/strategies/",  # Semantic strategy namespace에서 검색
                top_k=5,
                max_results=10,
            )

            if not results:
                return "No relevant information found in long-term memory. This might be new information or the memory extraction may still be processing."

            # Agent용 결과 형식 지정
            output = "📚 Retrieved from long-term memory:\\n\\n"
            for i, result in enumerate(results, 1):
                # MemoryRecord object의 content attribute에 액세스
                content = getattr(result, "content", str(result))
                # 매우 긴 content 자르기
                if len(content) > 300:
                    content = content[:300] + "..."
                output += f"{i}. {content}\\n\\n"

            return output

        except Exception as e:
            return f"⚠️ Error searching memory: {str(e)}. Proceeding without historical context."

    return FunctionTool.from_defaults(fn=search_long_term_memory)


# Memory 검색 tool 생성
memory_search_tool = create_memory_retrieval_tool(memory_id, "academic-researcher", region)

# Tool 목록에 memory 검색 추가
research_tools_with_memory = research_tools + [memory_search_tool]

print(f"✅ Memory retrieval tool created! Total tools: {len(research_tools_with_memory)}")
print("   Using namespace: /strategies/ (for semantic strategy compatibility)")

## 3c단계: Memory 구성 확인

Semantic strategy가 올바르게 구성되었는지 확인합니다.

In [ ]:
# Memory 구성 확인
memory_info = memory_manager.get_memory(memory_id)
print(f"Strategies: {memory_info.get('strategies')}")
print(f"Status: {memory_info.get('status')}")
print(f"Name: {memory_info.get('name')}")

# Strategy 세부 정보 표시
strategies = memory_info.get("strategies", [])
for strategy in strategies:
    print("\nStrategy Details:")
    print(f"  Name: {strategy.get('name')}")
    print(f"  Type: {strategy.get('type')}")
    print(f"  Status: {strategy.get('status')}")
    print(f"  ID: {strategy.get('strategyId')}")

## 4단계: Multi-Session Agent 구현

서로 다른 연구 session을 시뮬레이션하는 helper function을 생성합니다.

In [ ]:
# 장기 메모리 구성 (session 간)
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
RESEARCHER_ID = "academic-researcher"  # 모든 session에서 동일한 연구자


def create_research_session(session_name: str):
    """장기 메모리가 유지되는 새 연구 세션을 생성합니다."""
    context = AgentCoreMemoryContext(
        actor_id=RESEARCHER_ID,  # 동일한 연구자
        memory_id=memory_id,  # 동일한 memory store (장기 메모리 활성화)
        session_id=f"research-{session_name}",  # 기간별로 다른 session
        namespace="/academic-research/",
    )

    memory = AgentCoreMemory(context=context)
    llm = BedrockConverse(model=MODEL_ID)
    agent = FunctionAgent(
        tools=research_tools_with_memory,  # Memory 검색 기능이 있는 tool 사용
        llm=llm,
        verbose=True,  # Memory 검색 시점을 확인하도록 verbose 활성화
        system_prompt="""You are a senior research assistant with access to long-term memory.
        
CRITICAL: When asked about previous research, papers, findings, or historical information, 
you MUST use the search_long_term_memory tool FIRST before responding.

For example:
- "What research am I working on?" → Use search_long_term_memory("research topics")
- "What papers have I reviewed?" → Use search_long_term_memory("papers authors")
- "What findings do I have?" → Use search_long_term_memory("research findings")

Always provide conclusive, complete responses without asking follow-up questions.\n
Execute all requested actions immediately and completely. Provide detailed, professional responses.""",
    )

    return agent, memory


print("✅ Multi-session Academic Research Assistant setup complete!")

## 5단계: 1주 차 연구 Session - 기반 구축

첫 번째 연구 session을 시작하고 기초 지식을 확립합니다.

In [ ]:
# === 1주 차 연구 SESSION ===
print("🗓️ === WEEK 1: FOUNDATION RESEARCH ===")

agent_week1, memory_week1 = create_research_session("week1")

# 연구 기반 확립
response = await agent_week1.run(
    "I'm Dr. Sarah Smith from MIT starting comprehensive research on 'Machine Learning in Healthcare Applications'. "
    "Track this with status 'Literature Review'. My goal is to publish a systematic review by year-end.",
    memory=memory_week1,
)

print("🎯 Week 1 Foundation:")
print(response)

In [ ]:
# 상세 metric이 포함된 기초 논문 추가
response = await agent_week1.run(
    "Save paper: 'Deep Learning for Medical Image Analysis' by Zhang et al (2023). "
    "Key findings: CNNs achieve 95.2% accuracy in chest X-ray diagnosis, 12% improvement over radiologists, "
    "trained on 100,000 images, 0.03 false positive rate.",
    memory=memory_week1,
)
print("📄 Week 1 Paper 1:", response)

response = await agent_week1.run(
    "Save paper: 'Transformers in Medical NLP' by Johnson et al (2023). "
    "Key findings: BERT achieves 89.1% F1-score in clinical note classification, "
    "struggles with rare diseases (<70% accuracy), excels at symptom extraction (94% precision).",
    memory=memory_week1,
)
print("📄 Week 1 Paper 2:", response)
# 정확도 metric 명시적 추적
await agent_week1.run(
    "Track research metrics: metric_type 'CNN Accuracy', value '95.2%', source 'Zhang et al 2023', period 'Week 1'.",
    memory=memory_week1,
)
await agent_week1.run(
    "Track research metrics: metric_type 'Radiologist Improvement', value '12%', source 'Zhang et al 2023', period 'Week 1'.",
    memory=memory_week1,
)

In [ ]:
# Semantic memory 처리 시간 확보
import asyncio

print("\n⏳ Waiting for semantic memory extraction and indexing...")
print("   (AgentCore processes conversational events in the background)")
await asyncio.sleep(90)  # Memory 추출 대기 시간 연장
print("✅ Memory processing complete - memories should now be searchable")

## 6단계: 2주 차 연구 Session - Session 간 Memory 테스트

장기 메모리 검색을 테스트하고 새 연구를 추가합니다.

In [ ]:
# === 2주 차 연구 SESSION ===
print("\n🗓️ === WEEK 2: EXPANSION (NEW SESSION) ===")

agent_week2, memory_week2 = create_research_session("week2")

# Session 간 메모리 회상 테스트
response = await agent_week2.run(
    "What research am I working on? What specific accuracy metrics have I found so far? Who are the key authors?",
    memory=memory_week2,
)

print("🧠 Week 2 Memory Test:")
print(response)
print("\n✅ Expected: ML in Healthcare, Zhang 95.2%, Johnson 89.1% F1-score")

In [ ]:
# 이전 지식을 기반으로 새 연구 추가
response = await agent_week2.run(
    "Save paper: 'Federated Learning in Healthcare' by Brown et al (2023). "
    "Key findings: Privacy-preserving ML enables multi-hospital collaboration, 87.3% accuracy across 15 hospitals, "
    "23% improvement in rare disease detection when hospitals collaborate.",
    memory=memory_week2,
)
print("📄 Week 2 New Paper:", response)

# Session 간 비교 분석 테스트
response = await agent_week2.run(
    "Compare the accuracy results: Zhang's CNNs vs Johnson's BERT vs Brown's federated learning. "
    "Which performs best and in what contexts?",
    memory=memory_week2,
)
print("📊 Week 2 Comparative Analysis:")
print(response)
print("\n✅ Expected: Zhang 95.2% (imaging), Johnson 89.1% (NLP), Brown 87.3% (federated)")

## 7단계: 3주 차 연구 Session - 분석 단계

연구를 진행하고 상세한 session 간 회상을 테스트합니다.

In [ ]:
# === 3주 차 연구 SESSION ===
print("\n🗓️ === WEEK 3: ANALYSIS PHASE ===")

agent_week3, memory_week3 = create_research_session("week3")

# 연구 상태 업데이트
response = await agent_week3.run(
    "Update my 'Machine Learning in Healthcare Applications' research status to 'Analysis Phase' "
    "with notes: 'Reviewed 3 key papers, identified performance patterns: imaging>NLP>federated learning'.",
    memory=memory_week3,
)
print("📊 Week 3 Status Update:", response)

# 상세한 session 간 회상 테스트
response = await agent_week3.run(
    "What evidence do I have for the claim that imaging tasks show highest ML performance in healthcare? "
    "Include specific numbers and authors.",
    memory=memory_week3,
)
print("🔍 Week 3 Evidence Query:")
print(response)
print("\n✅ Expected: Zhang et al CNNs 95.2% vs Johnson BERT 89.1% vs Brown federated 87.3%")

## 8단계: 1개월 차 연구 Session - 종합 단계

종합적인 지식 통합과 연구 통합을 테스트합니다.

In [ ]:
# === 1개월 차 연구 SESSION ===
print("\n🗓️ === MONTH 1: SYNTHESIS PHASE ===")

agent_month1, memory_month1 = create_research_session("month1")

# 연구 상태를 종합 단계로 업데이트
response = await agent_month1.run(
    "Update my 'Machine Learning in Healthcare Applications' research status to 'Synthesis Phase' "
    "with notes: 'Completed 3-week literature review, ready to synthesize findings into coherent framework'.",
    memory=memory_month1,
)
print("📊 Month 1 Status Update:", response)

# 전체 주차에 걸친 종합적 통합 테스트
response = await agent_month1.run(
    "Based on all my research so far, what is the overall performance ranking of ML approaches in healthcare? "
    "Include all specific metrics and create a comprehensive comparison.",
    memory=memory_month1,
)
print("🔍 Month 1 Comprehensive Synthesis:")
print(response)
print("\n✅ Expected: Ranking with Zhang 95.2% > Johnson 89.1% > Brown 87.3%, domain analysis")

## 9단계: 2개월 차 연구 Session - 작성 단계

종합적인 회상 및 semantic search 기능을 테스트합니다.

In [ ]:
# === 2개월 차 연구 SESSION ===
print("\n🗓️ === MONTH 2: WRITING PHASE ===")

agent_month2, memory_month2 = create_research_session("month2")

# 작성을 위한 종합적인 회상 테스트
response = await agent_month2.run(
    "I'm writing my systematic review paper. What are ALL the papers I've reviewed with their exact accuracy metrics? "
    "I need this for my results table.",
    memory=memory_month2,
)
print("📝 Month 2 Comprehensive Recall:")
print(response)
print("\n✅ Expected: Zhang 95.2%, Johnson 89.1%, Brown 87.3% with full details")

In [ ]:
# 전체 연구 기록의 semantic search 테스트
response = await agent_month2.run(
    "What do I know about rare disease detection in my research? Which papers and what specific results?",
    memory=memory_month2,
)
print("🔍 Month 2 Semantic Search:")
print(response)
print("\n✅ Expected: Johnson <70% for rare diseases, Brown 23% improvement with collaboration")

## 10단계: 3개월 차 연구 Session - 연구비 제안서 시나리오

축적된 지식의 실질적인 활용을 테스트합니다.

In [ ]:
# === 3개월 차 연구 SESSION ===
print("\n🗓️ === MONTH 3: GRANT PROPOSAL ===")

agent_month3, memory_month3 = create_research_session("month3")

# 연구비 제안서 근거 수집
response = await agent_month3.run(
    "I'm writing an NIH grant proposal for $2M funding. What evidence can I cite about ML effectiveness in healthcare? "
    "I need specific numbers, authors, years, and sample sizes.",
    memory=memory_month3,
)
print("💰 Month 3 Grant Evidence:")
print(response)
print("\n✅ Expected: Comprehensive citation with Zhang 95.2% (100K images), Johnson 89.1%, Brown 87.3% (15 hospitals)")

In [ ]:
# 상세 milestone으로 연구 발전 과정 추적 테스트
response = await agent_month3.run(
    "Provide a detailed timeline of my research evolution from Week 1 to now. What specific milestones, "
    "metrics, and insights did I achieve each period? How did my research questions evolve?",
    memory=memory_month3,
)
print("📈 Month 3 Research Evolution:")
print(response)
print("\n✅ Expected: Week-by-week progression with specific milestones, metrics (95.2%, 89.1%, 87.3%), and insights")

## 11단계: 최종 Portfolio 평가

장기 메모리 기능을 종합적으로 테스트합니다.

In [ ]:
# 최종 종합 portfolio 질의
response = await agent_month3.run(
    "Provide my complete research portfolio: all topics I'm working on, all papers with metrics, "
    "all findings, current status of each project, and how they interconnect.",
    memory=memory_month3,
)
print("📋 Complete Research Portfolio:")
print(response)
print("\n✅ Expected: Full research history with all metrics, connections between ML healthcare topics")

## 🧪 자동 테스트 검증
이 셀을 실행하여 메모리 통합이 올바르게 작동하는지 검증합니다.

In [ ]:
# Validation function을 inline으로 정의
class TestValidator:
    def __init__(self):
        self.results = {}

    def validate_memory_recall(self, response):
        """에이전트가 세션 앞부분의 정보를 기억하는지 확인합니다."""
        # "I don't know"만 반환한 것이 아닌 실질적인 응답인지 확인
        has_content = len(response) > 50
        # Memory indicator 확인
        has_memory_indicators = any(
            word in response.lower()
            for word in [
                "earlier",
                "mentioned",
                "discussed",
                "previously",
                "you",
                "we",
                "our",
            ]
        )
        return "✅ PASS" if (has_content and has_memory_indicators) else "❌ FAIL"

    def validate_session_memory(self, response):
        """에이전트가 세션 내 컨텍스트를 유지하는지 확인합니다."""
        has_memory_content = len(response) > 100 and any(
            word in response.lower()
            for word in [
                "previous",
                "earlier",
                "mentioned",
                "discussed",
                "before",
                "already",
            ]
        )
        return "✅ PASS" if has_memory_content else "❌ FAIL"

    def validate_cross_reference(self, response):
        """에이전트가 현재 질의를 이전 컨텍스트와 연결할 수 있는지 확인합니다."""
        # 연결 표현 확인
        connecting_words = [
            "relate",
            "connection",
            "previous",
            "earlier",
            "discussed",
            "mentioned",
            "context",
            "based on",
            "as we",
            "as i",
        ]
        has_connection = any(word in response.lower() for word in connecting_words)
        has_substance = len(response) > 80
        return "✅ PASS" if (has_connection and has_substance) else "❌ FAIL"

    def run_validation_summary(self, test_results):
        print("🧪 COMPREHENSIVE TEST VALIDATION SUMMARY")
        print("=" * 60)

        total_tests = len(test_results)
        passed_tests = sum(1 for result in test_results.values() if "PASS" in result)
        pass_rate = (passed_tests / total_tests * 100) if total_tests > 0 else 0

        for test_name, result in test_results.items():
            print(f"{test_name}: {result}")

        print("=" * 60)
        print(f"📊 Overall Pass Rate: {passed_tests}/{total_tests} ({pass_rate:.1f}%)")

        if pass_rate >= 80:
            print("✅ EXCELLENT: Memory integration working correctly!")
        elif pass_rate >= 60:
            print("⚠️  GOOD: Most memory features working, some issues to investigate")
        else:
            print("❌ NEEDS ATTENTION: Memory integration has significant issues")

        return pass_rate


validator = TestValidator()
print("✅ Validation functions loaded!")

In [ ]:
# 모든 validation test 실행
test_results = {}

# 테스트 1: Memory 회상 - 에이전트가 논의 내용을 기억하는가?
response1 = await agent_month3.run("What have we discussed so far in this session?", memory=memory_month3)
test_results["Memory Recall"] = validator.validate_memory_recall(str(response1))
print(f"Response 1 length: {len(str(response1))} chars")

# 테스트 2: Session memory - 에이전트가 맥락을 유지하는가?
response2 = await agent_month3.run("What did we talk about earlier?", memory=memory_month3)
test_results["Session Memory"] = validator.validate_session_memory(str(response2))
print(f"Response 2 length: {len(str(response2))} chars")

# 테스트 3: 상호 참조 기능 - 이전 맥락과 연결할 수 있는가?
response3 = await agent_month3.run("How does this relate to what we discussed before?", memory=memory_month3)
test_results["Cross Reference"] = validator.validate_cross_reference(str(response3))
print(f"Response 3 length: {len(str(response3))} chars")

# 결과 표시
validator.run_validation_summary(test_results)

## 요약

이 Notebook에서는 다음 내용을 살펴봤습니다.

✅ **장기 메모리 통합**: LlamaIndex와 AgentCore Memory를 사용하여 session 간 지속성 구현

✅ **누적 지식 구축**: 수주 및 수개월에 걸쳐 연구 지식 축적

✅ **Semantic 검색**: 도우미가 여러 session에서 개념을 기반으로 관련 정보 검색

✅ **연구 발전 과정 추적**: 문헌 검토에서 분석과 작성까지 자연스러운 진행

✅ **Session 간 종합**: 여러 연구 session의 결과와 insight 연결

✅ **실질적인 활용**: 연구비 제안서 지원 및 종합 portfolio 관리

학술 연구 도우미는 장기 메모리를 통해 전체 연구 기록을 유지하고 장기 research project 전반에서 정교한 지식 검색을 지원하며 시간이 지날수록 더 똑똑해지는 지속적인 연구 동반자로 발전할 수 있음을 보여 줍니다.

## 정리

이 Notebook에서 사용한 resource를 정리하도록 memory를 삭제하겠습니다.

**참고**: Memory를 영구 삭제하려는 경우에만 실행하세요. memory_id 변수에는 이 Notebook의 앞부분에서 생성한 memory의 ID가 있어야 합니다.

In [ ]:
# AgentCore Memory resource 정리
try:
    from bedrock_agentcore.memory import MemoryClient

    client = MemoryClient(region_name=region)
    client.delete_memory(memory_id)
    print(f"✅ Successfully deleted memory: {memory_id}")

except NameError as e:
    print(f"⚠️  Variable not defined: {e}")
    print("Run the notebook from the beginning or set variables manually:")
    print("# memory_id = 'your-memory-id-here'")
    print("# region = 'us-east-1'")
except Exception as e:
    print(f"❌ Error deleting memory: {e}")